[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NTU-CompHydroMet-Lab/Datacube-demo/blob/main/ODC/notebooks/01_odc_load_demo.ipynb)

# Open Data Cube   
The Open Data Cube (ODC) is an open source solution for accessing, managing, and analyzing large quantities of Geographic Information System (GIS) data - namely Earth Observation (EO) data. It presents a common analytical framework composed of a series of data structures and tools which facilitate the organization and analysis of large gridded data collections.

Open Data Cube 是一個開源的觀測資料管理與分析框架。它可用來管理大量地球觀測資料，並透過Python API來進行資料存取、快速查詢與全球尺度的資料分析。

![image-2.png](attachment:image-2.png)
Source: [ODC_overview](https://www.opendatacube.org/overview-draft)

## ODC Sentinel-2 Land-Cover Load Demo

This notebook loads a fixed Greater Taipei bounding box from the indexed Taiwan Sentinel-2-derived annual land-cover GeoTIFFs.

`x` and `y` are projected raster coordinates in EPSG:32651. The plot below relabels ticks as longitude and latitude for readability. The land-cover class codes are stored in the `classification` data variable. NoData is class `0`.

## Colab Setup

This section prepares a self-contained ODC environment inside the Colab VM:

1. Install and start PostgreSQL, then create the `datacube` database and user.
2. Install the `datacube` Python package (same version as the local Docker demo).
3. Clone this repository — Git LFS pulls the demo GeoTIFFs (~370 MB, takes a few minutes).
4. Initialize the ODC schema, add the `s2_landcover_taiwan` product, and index the demo datasets.

The full setup takes roughly 3–5 minutes on a fresh Colab runtime.

All setup cells are guarded by `IN_COLAB`, so they are safe no-ops when this notebook
runs in the local Docker environment (`odc_local_demo`), where the index is already
built by `setup_odc_demo.sh`.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

In [ ]:
if IN_COLAB:
    # PostgreSQL backs the ODC index; git-lfs is needed to pull the demo GeoTIFFs.
    !sudo apt-get -qq update
    !sudo apt-get -qq install -y postgresql git-lfs
    !sudo service postgresql start
    !sudo -u postgres psql -tc "SELECT 1 FROM pg_roles WHERE rolname='datacube'" | grep -q 1 || sudo -u postgres psql -c "CREATE USER datacube WITH PASSWORD 'datacube';"
    !sudo -u postgres psql -tc "SELECT 1 FROM pg_database WHERE datname='datacube'" | grep -q 1 || sudo -u postgres createdb -O datacube datacube
    %pip install -q datacube==1.8.19 psycopg2-binary==2.9.9 pyyaml

In [ ]:
if IN_COLAB:
    import os

    REPO_DIR = "/content/Datacube-demo"
    DEMO_ROOT = f"{REPO_DIR}/ODC/odc_local_demo"

    if not os.path.exists(REPO_DIR):
        !git lfs install --skip-repo
        !git clone https://github.com/NTU-CompHydroMet-Lab/Datacube-demo.git {REPO_DIR}

    # Same connection settings that setup_odc_demo.sh writes inside the Docker container.
    # Write the config to a known, universally accessible location like /tmp.
    DATACUBE_CONFIG_FILE = "/tmp/.datacube.conf"
    with open(DATACUBE_CONFIG_FILE, "w") as f:
        f.write(
            "[datacube]\n"
            "db_hostname: localhost\n"
            "db_database: datacube\n"
            "db_username: datacube\n"
            "db_password: datacube\n"
        )

    # Set the DATACUBE_CONFIG_PATH environment variable for Python processes
    # and for the shell commands that follow.
    os.environ['DATACUBE_CONFIG_PATH'] = DATACUBE_CONFIG_FILE

    # Execute datacube commands as the postgres system user.
    # Pass DATACUBE_CONFIG_PATH as an environment variable to each command executed by sudo.

    # Run datacube system init as the postgres system user,
    # and override the DB_USERNAME environment variable to 'postgres'
    # so it connects with superuser privileges to create the schema.
    !sudo -u postgres DATACUBE_CONFIG_PATH={DATACUBE_CONFIG_FILE} DB_USERNAME=postgres datacube system init

    # Grant privileges to the 'datacube' user on the 'agdc' schema and its objects.
    # This ensures the 'datacube' user can access objects created by 'postgres' user during init.
    !sudo -u postgres psql -d datacube -c "GRANT USAGE ON SCHEMA agdc TO datacube;"
    !sudo -u postgres psql -d datacube -c "GRANT ALL PRIVILEGES ON ALL TABLES IN SCHEMA agdc TO datacube;"
    !sudo -u postgres psql -d datacube -c "ALTER DEFAULT PRIVILEGES IN SCHEMA agdc GRANT ALL PRIVILEGES ON TABLES TO datacube;"
    !sudo -u postgres psql -d datacube -c "GRANT ALL PRIVILEGES ON ALL SEQUENCES IN SCHEMA agdc TO datacube;"
    !sudo -u postgres psql -d datacube -c "ALTER DEFAULT PRIVILEGES IN SCHEMA agdc GRANT ALL PRIVILEGES ON SEQUENCES TO datacube;"

    # Verify system setup before proceeding.
    # Other commands can connect as the 'datacube' DB user, so no DB_USERNAME override is needed.
    !sudo -u postgres DATACUBE_CONFIG_PATH={DATACUBE_CONFIG_FILE} datacube system check

    !sudo -u postgres DATACUBE_CONFIG_PATH={DATACUBE_CONFIG_FILE} datacube product add {DEMO_ROOT}/products/s2_landcover_taiwan.yaml
    # The python script itself doesn't need the datacube config path, as it's not calling datacube CLI directly.
    !python {DEMO_ROOT}/scripts/write_dataset_yaml.py --data-dir {DEMO_ROOT}/data --dataset-dir {DEMO_ROOT}/datasets --product s2_landcover_taiwan --measurement classification
    !sudo -u postgres DATACUBE_CONFIG_PATH={DATACUBE_CONFIG_FILE} datacube dataset add {DEMO_ROOT}/datasets/*.yaml

In [ ]:
if IN_COLAB:
    # Sanity check: product indexed, datasets found, sample load succeeds.
    !DATACUBE_CONFIG_PATH={DATACUBE_CONFIG_FILE} python {DEMO_ROOT}/scripts/check_odc_demo.py

In [ ]:
from datacube import Datacube
from rasterio.warp import transform, transform_bounds
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
import matplotlib.patches as mpatches

PRODUCT = "s2_landcover_taiwan"
MEASUREMENT = "classification"
OUTPUT_CRS = "EPSG:32651"
RESOLUTION = (-10, 10)

# On Colab, Datacube() picks up the DATACUBE_CONFIG_PATH env var set in the setup cell.
dc = Datacube(app="01_odc_load_demo")

In [ ]:
dc.list_products().loc[[PRODUCT]]

In [ ]:
datasets = list(dc.index.datasets.search(product=PRODUCT))
len(datasets), datasets[:1]

## Taipei Bounding Box

The bbox below is a focused Taipei core area in lon/lat. It is transformed to EPSG:32651 before loading from ODC.

In [ ]:
# Greater Taipei core bbox: west, south, east, north in EPSG:4326
# Use a focused bbox for an interactive notebook; expand only when needed.
taipei_bbox_lonlat = (121.40, 24.95, 121.70, 25.20)

min_x, min_y, max_x, max_y = transform_bounds(
    "EPSG:4326",
    OUTPUT_CRS,
    *taipei_bbox_lonlat,
    densify_pts=21,
)

taipei_query = {
    "x": (min_x, max_x),
    "y": (min_y, max_y),
}
taipei_query

In [ ]:
taipei = dc.load(
    product=PRODUCT,
    measurements=[MEASUREMENT],
    x=taipei_query["x"],
    y=taipei_query["y"],
    crs=OUTPUT_CRS,
    output_crs=OUTPUT_CRS,
    resolution=RESOLUTION,
)
taipei

In [ ]:
classification = taipei[MEASUREMENT]
classification

## Area Summary

This avoids converting the full raster to a pandas Series. It counts each class directly with NumPy, then creates a small summary table.

In [ ]:
class_names = {
    1: "Water",
    2: "Trees",
    4: "Flooded Vegetation",
    5: "Crops",
    7: "Built Area",
    8: "Bare Ground",
    9: "Snow/Ice",
    10: "Clouds",
    11: "Rangeland",
}

pixel_area_m2 = 100
class_codes = list(class_names)

rows = []
for time_value in classification.time.values:
    arr = classification.sel(time=time_value).values
    for code in class_codes:
        pixel_count = int(np.count_nonzero(arr == code))
        rows.append(
            {
                "time": str(time_value)[:10],
                "class_code": code,
                "class_name": class_names[code],
                "pixel_count": pixel_count,
                "area_km2": pixel_count * pixel_area_m2 / 1_000_000,
            }
        )

area = pd.DataFrame(rows)
area

## Plot

The plot uses one time slice and decimates pixels for display only. The raster is still loaded in EPSG:32651, but axis tick labels are converted back to longitude and latitude.

In [ ]:
class_colors = {
    1: "#419BDF",
    2: "#397D49",
    4: "#7A87C6",
    5: "#E49635",
    7: "#C4281B",
    8: "#A59B8F",
    9: "#B39FE1",
    10: "#FFFFFF",
    11: "#E3E2C3",
}

codes = list(class_colors)
bounds = [0.5, 1.5, 2.5, 4.5, 5.5, 7.5, 8.5, 9.5, 10.5, 11.5]
cmap = ListedColormap([class_colors[code] for code in codes])
norm = BoundaryNorm(bounds, cmap.N)


def set_lonlat_ticks(ax, x_values, y_values, x_count=5, y_count=5):
    x_ticks = np.linspace(float(x_values.min()), float(x_values.max()), x_count)
    y_ticks = np.linspace(float(y_values.min()), float(y_values.max()), y_count)
    center_x = float(x_values.mean())
    center_y = float(y_values.mean())

    lon_labels, _ = transform(OUTPUT_CRS, "EPSG:4326", x_ticks.tolist(), [center_y] * len(x_ticks))
    _, lat_labels = transform(OUTPUT_CRS, "EPSG:4326", [center_x] * len(y_ticks), y_ticks.tolist())

    ax.set_xticks(x_ticks)
    ax.set_yticks(y_ticks)
    ax.set_xticklabels([f"{lon:.2f}" for lon in lon_labels])
    ax.set_yticklabels([f"{lat:.2f}" for lat in lat_labels])


plot_time_index = 0
display_step = 10

plot_data = classification.isel(time=plot_time_index)
plot_data = plot_data.where(plot_data != 0)
plot_data = plot_data.isel(y=slice(None, None, display_step), x=slice(None, None, display_step))

fig, ax = plt.subplots(figsize=(10, 9))
plot_data.plot.imshow(ax=ax, cmap=cmap, norm=norm, add_colorbar=False)
set_lonlat_ticks(ax, plot_data.x, plot_data.y)

time_value = str(classification.time.values[plot_time_index])[:10]
ax.set_title(f"Greater Taipei Land Cover - {time_value}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

patches = [
    mpatches.Patch(color=class_colors[code], label=f"{code} {class_names[code]}")
    for code in codes
]
ax.legend(handles=patches, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
plt.tight_layout()
plt.show()